# 20. Ensemble Learning: Random Forest

## Algorithm Category
**Type**: Ensemble Learning - Classification/Regression  
**Complexity**: Medium  
**Use Case**: Ensemble of decision trees with bagging and feature randomness

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Random Forest algorithm and ensemble learning principles
- Implement Random Forest for classification and regression
- Understand bagging, feature sampling, and out-of-bag evaluation
- Extract and interpret feature importance
- Tune hyperparameters (n_estimators, max_depth, max_features)
- Apply Random Forest to real-world problems

## Historical Context

Random Forest was developed by Leo Breiman in 2001:
- Breiman, L. (2001): "Random Forests"
- Combines bagging (Bootstrap Aggregating) with random feature selection
- Built on earlier work: Decision Trees (Breiman et al., 1984) and Bagging (Breiman, 1996)

**Key Papers/References:**
- Breiman, L. (2001). "Random Forests"
- Breiman, L. (1996). "Bagging Predictors"
- Ho, T.K. (1995). "Random Decision Forests"

## When to Use Random Forest

Random Forest is appropriate when:
- You need high accuracy with minimal tuning
- Working with mixed data types (numerical and categorical)
- Feature importance interpretation is needed
- Handling missing values and outliers
- Large datasets with many features
- Non-linear relationships in data

## Theory & Mechanics

### Mathematical Foundation

Random Forest combines multiple decision trees using bagging and random feature selection.

**Bagging (Bootstrap Aggregating):**
- Train each tree on a bootstrap sample (random sampling with replacement)
- Average predictions (regression) or majority vote (classification)

**Random Feature Selection:**
- At each split, consider only a random subset of features
- Typically: $\sqrt{n}$ features for classification, $n/3$ for regression

**Final Prediction:**
- **Classification**: $\hat{y} = \text{mode}(\{\hat{y}_1, \hat{y}_2, ..., \hat{y}_T\})$
- **Regression**: $\hat{y} = \frac{1}{T}\sum_{i=1}^{T}\hat{y}_i$

Where $T$ is the number of trees.

**Out-of-Bag (OOB) Score:**
- Each tree is trained on ~63% of data (bootstrap sample)
- Remaining ~37% is "out-of-bag" and can be used for validation
- OOB score estimates generalization without separate validation set

### How It Works

1. **Bootstrap Sampling**: Create T bootstrap samples from training data
2. **Train Trees**: Train a decision tree on each bootstrap sample
3. **Random Features**: At each split, consider random subset of features
4. **Aggregate**: Combine predictions from all trees
5. **OOB Evaluation**: Use out-of-bag samples for validation

### Key Hyperparameters

- **n_estimators**: Number of trees (more = better but slower)
- **max_depth**: Maximum depth of trees (None = unlimited)
- **max_features**: Number of features to consider for best split
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node
- **bootstrap**: Whether to use bootstrap sampling (default: True)
- **oob_score**: Whether to calculate out-of-bag score (default: False)

### Advantages

- Reduces overfitting compared to single decision tree
- Handles missing values and outliers well
- Provides feature importance
- No feature scaling required
- Works well with default parameters

### Limitations

- Less interpretable than single decision tree
- Can be memory intensive (stores all trees)
- Slower prediction than linear models
- May not perform well on very high-dimensional sparse data


## Implementation

Let's implement Random Forest for both classification and regression.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.tree import plot_tree

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix
from src.models.ensemble import extract_feature_importance, plot_feature_importance, calculate_oob_error
from src.utils.benchmarking import benchmark_model_training
from src.utils.traceability import save_traceability_data
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Classification Example: Breast Cancer dataset
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='Target')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {cancer.target_names.tolist()}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, oob_score=True)
model.fit(X_train, y_train)

print("\nRandom Forest Classifier:")
print(f"Number of trees: {model.n_estimators}")
print(f"Out-of-bag score: {model.oob_score_:.3f}")

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.3f}")

# Evaluate
results = evaluate_classifier(model, X_test, y_test)
metrics = calculate_classification_metrics(y_test.values, y_pred)
print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")


## Feature Importance

Let's extract and visualize feature importance.


In [ ]:
# Extract feature importance
feature_importance = extract_feature_importance(model, feature_names=X.columns.tolist())
print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize feature importance
plot_feature_importance(feature_importance, top_n=15, title="Random Forest Feature Importance")


## Validation & Testing

Let's validate the model and analyze out-of-bag error.


In [ ]:
# Validation 1: Out-of-bag error
oob_error = calculate_oob_error(model)
if oob_error is not None:
    print(f"Out-of-Bag Error: {oob_error:.3f}")
    print(f"Out-of-Bag Accuracy: {1 - oob_error:.3f}")

# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f"\nCross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")

# Validation 3: Check model output
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert accuracy > 0.5, "Accuracy should be better than random!"
print("\n✓ Validation checks passed")


## Effect of Number of Trees

Let's see how performance changes with the number of trees.


In [ ]:
# Test different numbers of trees
n_trees_range = [10, 25, 50, 100, 200, 300]
train_scores = []
test_scores = []
oob_scores = []

for n_trees in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n_trees, max_depth=10, 
                               random_state=42, oob_score=True, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    train_scores.append(accuracy_score(y_train, rf.predict(X_train)))
    test_scores.append(accuracy_score(y_test, rf.predict(X_test)))
    oob_scores.append(rf.oob_score_)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(n_trees_range, train_scores, 'o-', label='Training Accuracy')
plt.plot(n_trees_range, test_scores, 's-', label='Test Accuracy')
plt.plot(n_trees_range, oob_scores, '^-', label='OOB Accuracy')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest: Effect of Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(n_trees_range, [t - s for t, s in zip(train_scores, test_scores)], 'o-', color='red')
plt.xlabel('Number of Trees')
plt.ylabel('Train - Test Accuracy')
plt.title('Overfitting Gap')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best number of trees (by test accuracy): {n_trees_range[np.argmax(test_scores)]}")


## Regression Example

Let's apply Random Forest to regression.


In [ ]:
# Regression Example: Diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='Target')

X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(X_reg, y_reg, test_size=0.2, random_state=42)

# Train Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, oob_score=True)
rf_reg.fit(X_reg_train, y_reg_train)

y_reg_pred = rf_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("Random Forest Regression:")
print(f"  Test RMSE: {rmse:.3f}")
print(f"  Out-of-bag score: {rf_reg.oob_score_:.3f}")

# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.6)
plt.plot([y_reg_test.min(), y_reg_test.max()], 
         [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Random Forest Regression: Predicted vs Actual')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Real-World Application

Let's tune hyperparameters using GridSearchCV.


In [ ]:
# Hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': [2, 5, 10]
}

# Use smaller grid for faster execution
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, oob_score=True),
    param_grid,
    cv=3,  # Reduced folds for speed
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("\nBest Hyperparameters:")
print(grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")

# Evaluate best model
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, best_pred)
print(f"Test Accuracy with Best Model: {best_accuracy:.3f}")

# Compare with default model
print(f"\nComparison:")
print(f"  Default model accuracy: {accuracy:.3f}")
print(f"  Tuned model accuracy: {best_accuracy:.3f}")
print(f"  Improvement: {best_accuracy - accuracy:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Random Forest Basics**
   - Ensemble of decision trees using bagging
   - Random feature selection at each split
   - Reduces overfitting compared to single tree
   - Provides feature importance

2. **Bagging**
   - Bootstrap sampling (random sampling with replacement)
   - Each tree trained on different subset of data
   - Aggregates predictions (majority vote or average)

3. **Key Hyperparameters**
   - **n_estimators**: More trees = better but slower
   - **max_depth**: Controls tree complexity
   - **max_features**: Number of features to consider per split
   - **min_samples_split/leaf**: Prevents overfitting

4. **Out-of-Bag Evaluation**
   - ~37% of data not used in each tree's training
   - Can estimate generalization without separate validation set
   - Useful for model selection

### When to Use Random Forest

✅ **Good for:**
- High accuracy with minimal tuning
- Mixed data types (numerical and categorical)
- Feature importance interpretation
- Handling missing values and outliers
- Non-linear relationships
- Large datasets with many features

❌ **Not ideal for:**
- Very high-dimensional sparse data
- When interpretability is crucial (use single tree)
- Real-time predictions (can be slow)
- Linear relationships (use linear models)
- Very large datasets (memory intensive)

### Next Steps

- Try **Extra Trees (Extremely Randomized Trees)** for faster training
- Explore **Gradient Boosting** for sequential improvement
- Consider **XGBoost** for optimized tree boosting
- Use **Feature Selection** based on importance scores
